In [18]:
import torch
import torch.nn as nn

In [19]:
import torch
print(torch.__version__)

2.7.1


In [20]:

from torchvision import datasets, transforms
import torch.utils.data as dataload

In [21]:
import brevitas.nn as qnn
from brevitas.quant import Int8WeightPerTensorFloat   # signed, for weights
from brevitas.quant import Int8ActPerTensorFloat      # signed, for the input
from brevitas.quant import Uint8ActPerTensorFloat     # UNsigned, for ReLU outputs
from brevitas.quant import Int8Bias

In [22]:
from brevitas.export import export_qonnx

In [23]:
# start with BIT_WIDTH of 2, so it can store 4 values
BIT_WIDTH = 2


In [24]:
# The transforms we need:
# to tensor (0-1 values, permutes channels?)
data_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.1307], std=[0.3081])
])

In [25]:
# Now we import training data
training_data = datasets.MNIST(
    root="./data",
    # Give all training images
    train=True,
    # download
    download=True,
    # apply transforms
    transform=data_transforms
)

# Btw it downloads (training data + test data) * labels

In [26]:
training_loader = dataload.DataLoader(
    dataset=training_data,
    batch_size=64,
    shuffle=True, # Every epoch randomize batch order
    num_workers=4 # subprocess for parallel loading
)

In [27]:
# now deal with test data. Extract that.
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=data_transforms)
test_loader = dataload.DataLoader(
    dataset=test_data,
    batch_size=1000,
    shuffle=False
)


In [28]:
class QMNIST(nn.Module):
    def __init__(self):
        super().__init__() # Calling nn.module constructor
        # We are not too stingy with bits rn, can afford to use 8 bits
        # Int8Act... has negatives and positives
        self.inp_quant = qnn.QuantIdentity(bit_width=8, act_quant=Int8ActPerTensorFloat, return_quant_tensor=True)
        # filter values get quantized now
        # weight_quant (and weight_bit_width) makes it so that filters are quantized to 4 bits
        # be generous now (conv1 touches raw input pixels, don't compress that too much, don't throw away signal), so thats why 4 bits are used
        # also make sure bias is affected
        self.conv1 = qnn.QuantConv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1, weight_quant=Int8WeightPerTensorFloat, weight_bit_width=4, bias=True, bias_quant=Int8Bias)
        # does relu, snaps to 4 values! per tesnor (fgpa friendly)
        # this need to be unsigned, relu has no negative values
        self.relu1 = qnn.QuantReLU(act_quant=Uint8ActPerTensorFloat, bit_width=2, return_quant_tensor=True)
        # doesn't change anything
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Now we create 2nd convolution
        # Two bits this time (stingy in middle)
        self.conv2 = qnn.QuantConv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1, weight_quant=Int8WeightPerTensorFloat, weight_bit_width=2, bias=True, bias_quant=Int8Bias)
        # And now with 2 bit relu
        self.relu2 = qnn.QuantReLU(act_quant=Uint8ActPerTensorFloat, bit_width=2, return_quant_tensor=True)

        # Okay, but now we need to pool again.
        # As filters increase, the maps get smaller
        # Theortically you can reuse the pooling, but i don't realy want to
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Here we are using weight instead (because these are weights)
        # Centered around zero, unlike Act
        self.fc1 = qnn.QuantLinear(64 * 7 * 7, 128, weight_quant=Int8WeightPerTensorFloat, weight_bit_width=2, bias=True, bias_quant=Int8Bias)
        self.relu3 = qnn.QuantReLU(act_quant=Uint8ActPerTensorFloat, bit_width=2, return_quant_tensor=True)
        self.fc2 = qnn.QuantLinear(128, 10, weight_quant=Int8WeightPerTensorFloat, weight_bit_width=4, bias=True, bias_quant=Int8Bias)
        # No rlue (logits need ot go negative)

    def forward(self, x):
        # quantizes it
        x = self.inp_quant(x)
        x = self.pool(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))

        x = x.view(x.size(0), -1)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x






In [29]:
# BROOOO we can't use MPS, brevitas uses unusual operations...
device = torch.device('cuda' if torch.cuda.is_available()
                       else 'cpu')
# moves model onto chosen device
model = QMNIST().to(device)
print(model)

QMNIST(
  (inp_quant): QuantIdentity(
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (act_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (fused_activation_quant_proxy): FusedActivationQuantProxy(
        (activation_impl): Identity()
        (tensor_quant): RescalingIntQuant(
          (int_quant): IntQuant(
            (float_to_int_impl): RoundSte()
            (tensor_clamp_impl): TensorClamp()
            (delay_wrapper): DelayWrapper(
              (delay_impl): _NoDelay()
            )
            (input_view_impl): Identity()
          )
          (scaling_impl): ParameterFromRuntimeStatsScaling(
            (stats_input_view_shape_impl): OverTensorView()
            (stats): _Stats(
              (stats_impl): AbsPercentile()
            )
            (abs_value): _AbsValue(
              (apply_abs): Abs()
            )
            (restrict_scaling): _RestrictValue(
              (res

In [30]:
# Applies softmax function, as well as does loss function stuff
criterion = nn.CrossEntropyLoss()
# now the optimizer: the actual traning model
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

In [31]:
# training loop
num_epochs = 10

for epoch in range(num_epochs):
    # gets it in model mode, doesn't do anything
    model.train()
    # gives 64 images, runnings about 937 times
    running_loss = 0.0
    for images, labels in training_loader:
        # moves onto same device model
        images, labels = images.to(device), labels.to(device)
        # reset gradients
        optimizer.zero_grad()
        outputs = model(images)
        # finds loss. applies softmax function
        loss = criterion(outputs, labels)
        # Back propogation: calculate gradient for every weight
        # the insane step..
        loss.backward()
        # Now makes a step (updates all weights)
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(training_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.10f}")
torch.save(model.state_dict(), 'mnist_cnn.pth')


Epoch 1/10, Loss: 0.1847602740
Epoch 2/10, Loss: 0.0700502494
Epoch 3/10, Loss: 0.0511617808
Epoch 4/10, Loss: 0.0414734766
Epoch 5/10, Loss: 0.0343153812
Epoch 6/10, Loss: 0.0297387337
Epoch 7/10, Loss: 0.0256123224
Epoch 8/10, Loss: 0.0224228793
Epoch 9/10, Loss: 0.0194600411
Epoch 10/10, Loss: 0.0178174871


In [32]:
model.eval() # locks in scale values, all parameters
model.cpu()  # moves onto cpu
# 1 batch, 1 channel (greyscale), 28 x 28
input_shape = (1, 1, 28, 28)
export_file = "mnist_finn_ready.onnx"
# Apparently the way it works by running a fake foward pass (tracing)... and needs the right shape to be pushed through
export_qonnx(model, torch.randn(input_shape), export_file)   # 4. trace + write
print(f"Model exported to {export_file}")


Model exported to mnist_finn_ready.onnx


In [34]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 98.75%
